# 🚀 JEE-LLM — Inference Demo

This notebook demonstrates how to use JEE-LLM for solving JEE problems interactively.
Great for daily practice!

**Prerequisites**: Complete at least Stage 1 (SFT) training and have a checkpoint.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from IPython.display import Markdown, display

# ── Configuration ─────────────────────────────────────────────
MODEL_PATH = '../checkpoints/jee-llm-v1'   # Change to your checkpoint
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LOAD_IN_4BIT = DEVICE == 'cuda'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Load Model ────────────────────────────────────────────────
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model...')
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map='auto',
        attn_implementation='flash_attention_2',
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, torch_dtype=torch.float32
    )

model.eval()
print('✅ Model loaded!')

In [ ]:
# ── Solver function ───────────────────────────────────────────
SYSTEM_PROMPT = """You are JEE-LLM, an elite IIT-JEE Advanced tutor.
Solve problems step-by-step with complete mathematical rigor.
Always present the final answer inside \\boxed{}."""

def solve(
    question: str,
    subject: str = 'Auto',
    topic: str = '',
    question_type: str = 'single_correct',
    options: dict = None,
    max_new_tokens: int = 1024,
    temperature: float = 0.7,
    render_markdown: bool = True,
):
    """Solve a JEE problem and display the solution."""
    ctx_parts = []
    if subject != 'Auto':
        ctx_parts.append(f'Subject: {subject}')
    if topic:
        ctx_parts.append(f'Topic: {topic}')
    ctx_parts.append(f'Type: {question_type}')
    ctx = ' | '.join(ctx_parts)

    user_content = f'[{ctx}]\n\n{question}'
    if options:
        opts_str = '\n'.join([f'({k}) {v}' for k, v in options.items()])
        user_content += f'\n\nOptions:\n{opts_str}'

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    import time
    t0 = time.time()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=max(temperature, 1e-6),
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.time() - t0
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    n_tokens = len(generated_ids)

    output_md = f"""
## 📝 Question
{question}

---

## 💡 Solution
{response}

---
*⏱️ {elapsed:.1f}s · {n_tokens} tokens · {n_tokens/elapsed:.0f} tok/s*
"""

    if render_markdown:
        display(Markdown(output_md))
    else:
        print(response)

    return response

print('✅ solve() function ready!')

In [ ]:
# ── Example 1: Mathematics — Integral Calculus ────────────────
solve(
    question=r'Evaluate: $\int_0^{\pi} x \cdot \sin(x)\, dx$',
    subject='Mathematics',
    topic='Integral Calculus',
    question_type='numerical',
)

In [ ]:
# ── Example 2: Physics — MCQ ──────────────────────────────────
solve(
    question=(
        'A particle moves in a circle of radius R = 2 m with constant angular '
        'velocity ω = 3 rad/s. The magnitude of the centripetal acceleration is:'
    ),
    subject='Physics',
    topic='Circular Motion',
    question_type='single_correct',
    options={'A': '6 m/s²', 'B': '9 m/s²', 'C': '18 m/s²', 'D': '3 m/s²'},
)

In [ ]:
# ── Example 3: Chemistry ──────────────────────────────────────
solve(
    question=(
        'For the reaction: N₂(g) + 3H₂(g) ⇌ 2NH₃(g), '
        'if Kp = 9.25 × 10⁻² atm⁻² at 500°C, '
        'find Kc at the same temperature. (R = 0.0821 L·atm·mol⁻¹·K⁻¹)'
    ),
    subject='Chemistry',
    topic='Chemical Equilibrium',
    question_type='numerical',
)

In [ ]:
# ── Interactive: Your own question ───────────────────────────
# Change this to any JEE problem you want to practice!
YOUR_QUESTION = """
If the sum of first n terms of an AP is 3n² + 5n, find the 20th term.
"""

solve(
    question=YOUR_QUESTION.strip(),
    subject='Mathematics',
    topic='Sequences and Series',
    question_type='numerical',
)